# Activity 6 — Databases (MongoDB)
**Module:** Advanced Programming — Week 2  
**University of York, MSc Computer Science**

**Dataset:** `People.json` — 5 student records  
**Database:** MongoDB (document database)

---
## Setup

### Prerequisites
1. Install MongoDB Community Edition (see York Study Skills module for instructions)
2. Install the Python driver:
```bash
pip install pymongo
```
3. Start the MongoDB service before running these cells:
```bash
# macOS (Homebrew)
brew services start mongodb-community

# Linux / WSL
sudo systemctl start mongod

# Windows
net start MongoDB
```

---
## Step 1: Connect and load data

In [ ]:
import json
from pymongo import MongoClient

# Connect to local MongoDB instance
client = MongoClient('mongodb://localhost:27017/')

# Create (or access) a database and collection
db = client['york_advanced_programming']
collection = db['students']

# Drop the collection first to avoid duplicate inserts on re-run
collection.drop()

# Load the JSON file
with open('People.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Insert all student documents
result = collection.insert_many(data['students'])

print(f"Inserted {len(result.inserted_ids)} documents")
print(f"IDs: {result.inserted_ids}")

---
## Step 2: Verify the data loaded correctly

In [ ]:
print("All documents in collection:")
print()
for doc in collection.find({}, {'_id': 0}):
    print(doc)
    print()

---
## Query 1: Full name of anyone over 25

**MongoDB query logic:**  
`{ "age": { "$gt": 25 } }` — field `age` greater than 25  

**Projection:** return `fullName` fields only, exclude MongoDB's `_id`

In [ ]:
print("=== Query 1: Full name of anyone over 25 ===")
print()

query = {"age": {"$gt": 25}}
projection = {"_id": 0, "fullName": 1, "age": 1}

results = collection.find(query, projection)

for doc in results:
    fn = doc['fullName']
    # Build full name: Title First [Middle...] Surname
    middle_names = [m for m in fn.get('other', []) if m is not None]
    parts = [fn['title'], fn['first']] + middle_names + [fn['surname']]
    full_name = ' '.join(parts)
    print(f"  Age {doc['age']:>3}: {full_name}")

print()
print("Note: age > 25 excludes Takeshi Tanaka (25) and Iolanda Melo (23).")

---
## Query 2: ID of anyone who does not have any middle names

**Challenge:** In this dataset, the `other` field is always present as a list. A student with no middle names has `"other": [null]` — a list containing a single null element, not an empty list.

**Query logic:**  
Find documents where `fullName.other` contains only null values — meaning no actual names.

We use `$not` combined with `$elemMatch` to find documents where there is no non-null element in the `other` array.

In [ ]:
print("=== Query 2: ID of anyone with no middle names ===")
print()

# Strategy: find documents where fullName.other has NO non-null elements
# $not $elemMatch with $ne: null means "no element that is not null"
query = {
    "fullName.other": {
        "$not": {
            "$elemMatch": {"$ne": None}
        }
    }
}
projection = {"_id": 0, "id": 1, "fullName.first": 1, "fullName.surname": 1, "fullName.other": 1}

results = collection.find(query, projection)

for doc in results:
    fn = doc['fullName']
    print(f"  ID: {doc['id']}  — {fn['first']} {fn['surname']}")
    print(f"       (other field: {fn['other']})")

print()
print("Note: Takeshi Tanaka has \"other\": [null] — no actual middle name.")

---
## Query 3: Count men and women (separately) not living in Tokyo

**Gender logic:**  
Gender is not an explicit field in the dataset — it is inferred from the `title` field:
- `Mr` → male
- `Mrs`, `Miss`, `Ms` → female

**Query logic:**  
Filter: `city != "Tokyo"`, then group by title category (male/female).

In [ ]:
print("=== Query 3: Count men and women not living in Tokyo ===")
print()

# --- Approach A: Python-side aggregation (simple, readable) ---
not_tokyo = list(collection.find(
    {"city": {"$ne": "Tokyo"}},
    {"_id": 0, "fullName.title": 1, "fullName.first": 1, "fullName.surname": 1, "city": 1}
))

male_titles   = {"Mr"}
female_titles = {"Mrs", "Miss", "Ms"}

men   = [d for d in not_tokyo if d['fullName']['title'] in male_titles]
women = [d for d in not_tokyo if d['fullName']['title'] in female_titles]

print(f"Not in Tokyo — Men   ({len(men)}):")
for d in men:
    fn = d['fullName']
    print(f"    {fn['title']} {fn['first']} {fn['surname']} — {d['city']}")

print(f"\nNot in Tokyo — Women ({len(women)}):")
for d in women:
    fn = d['fullName']
    print(f"    {fn['title']} {fn['first']} {fn['surname']} — {d['city']}")

print(f"\nFinal count: {len(men)} man/men, {len(women)} woman/women not living in Tokyo")

In [ ]:
# --- Approach B: MongoDB aggregation pipeline (server-side, more scalable) ---
# This is the preferred approach for large datasets — filtering and grouping happens
# inside the database rather than loading all documents into Python memory.

print("=== Query 3 (Aggregation Pipeline) ===")
print()

pipeline = [
    # Stage 1: Exclude students in Tokyo
    {"$match": {"city": {"$ne": "Tokyo"}}},

    # Stage 2: Add a 'gender' field based on title
    {"$addFields": {
        "gender": {
            "$cond": {
                "if": {"$eq": ["$fullName.title", "Mr"]},
                "then": "male",
                "else": "female"
            }
        }
    }},

    # Stage 3: Group by gender and count
    {"$group": {
        "_id": "$gender",
        "count": {"$sum": 1}
    }},

    # Stage 4: Sort for consistent output
    {"$sort": {"_id": 1}}
]

agg_results = list(collection.aggregate(pipeline))

for result in agg_results:
    print(f"  {result['_id'].capitalize()}: {result['count']}")

---
## Expected results summary

Given the dataset (5 students: Lisa/London, Lorenzo/Paris, Takeshi/Tokyo, Tanveer/Mumbai, Iolanda/Lisbon):

| Query | Expected result |
|---|---|
| Full name of anyone over 25 | Mrs Lisa Melanie Penny (32), Mr Lorenzo Ruelle Garlen Dubois (38), Mr Tanveer Vihaan Patel (27) |
| ID of anyone with no middle names | 546854 (Takeshi Tanaka — `other: [null]`) |
| Men not in Tokyo | 2 — Lorenzo Dubois (Paris), Tanveer Patel (Mumbai) |
| Women not in Tokyo | 2 — Lisa Penny (London), Iolanda Melo (Lisbon) |

---
## Design notes

### Why MongoDB for this data?
The `People.json` structure is a good fit for a document database because each student has a nested sub-document (`fullName`) and a variable-length list (`other`). In a relational database (SQLite, PostgreSQL), this would require either a separate `middle_names` table (normalised) or a serialised string column — both less natural than a document store.

### Null handling in Query 2
The `$not $elemMatch $ne null` pattern is a common MongoDB idiom for "array contains only null values." It is more robust than checking `{"fullName.other": [null]}` directly because the latter requires an exact array match including order, which would fail if the data later gains multiple null entries.

### Aggregation pipeline vs Python-side filtering
For 5 documents, both approaches produce identical results. At scale (thousands of students), the aggregation pipeline is significantly faster because:
- Filtering happens inside the database engine before data crosses the network
- MongoDB can use indexes on the `city` and `fullName.title` fields
- Python receives only the final counts, not all matching documents